# The SIR Epidemic Model
**Category: Biology**

## 1. Introduction

The SIR model, introduced by Kermack and McKendrick in 1927, is the foundational compartmental model in mathematical epidemiology. It partitions a population into three groups — **Susceptible**, **Infectious**, and **Recovered** — and uses a system of ODEs to describe the flow between them. Despite its simplicity, it captures the essential dynamics of epidemic waves and gives rise to one of the most influential concepts in public health: the **basic reproduction number** $\mathcal{R}_0$.

## 2. Model Formulation

### 2.1 Equations

Let $S(t)$, $I(t)$, $R(t)$ be the fractions of the population in each compartment, so $S + I + R = 1$. The dynamics are
$$
\frac{dS}{dt} = -\beta S I
$$
$$
\frac{dI}{dt} = \beta S I - \gamma I
$$
$$
\frac{dR}{dt} = \gamma I
$$
where $\beta$ is the transmission rate (contacts per unit time $\times$ probability of transmission per contact) and $\gamma = 1/D$ is the recovery rate, with $D$ the mean infectious period.

### 2.2 The Basic Reproduction Number

The **basic reproduction number** is
$$
\mathcal{R}_0 = \frac{\beta}{\gamma}
$$
It represents the average number of secondary infections produced by one infectious individual in a fully susceptible population. The **epidemic threshold theorem** states:
- If $\mathcal{R}_0 > 1$: the infection spreads ($dI/dt > 0$ initially when $S \approx 1$)
- If $\mathcal{R}_0 \leq 1$: the infection dies out

The epidemic peaks when $S = 1/\mathcal{R}_0$, and the **herd immunity threshold** is $1 - 1/\mathcal{R}_0$.

### 2.3 Final Epidemic Size

The total fraction of the population ultimately infected, $R_\infty$, satisfies the transcendental equation
$$
R_\infty = 1 - e^{-\mathcal{R}_0\, R_\infty}
$$
This has no closed form but is easily solved numerically. For $\mathcal{R}_0 = 2.5$ (roughly measles before vaccination), $R_\infty \approx 0.893$ — nearly $90\%$ of the population.

## 3. Numerical Simulation

### 3.1 Epidemic Curves for Various $\mathcal{R}_0$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

def sir(t, y, beta, gamma):
    S, I, R = y
    return [-beta*S*I, beta*S*I - gamma*I, gamma*I]

gamma = 1/10  # 10-day infectious period
R0_values = [1.5, 2.0, 2.5, 3.5]
I0 = 1e-4
t_span = (0, 300)
t_eval = np.linspace(*t_span, 3000)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for R0 in R0_values:
    beta = R0 * gamma
    sol = solve_ivp(sir, t_span, [1 - I0, I0, 0], args=(beta, gamma),
                    t_eval=t_eval, rtol=1e-9)
    axes[0].plot(sol.t, sol.y[1] * 100, label=f'$\\mathcal{{R}}_0 = {R0}$', linewidth=2)

axes[0].set_xlabel('Days')
axes[0].set_ylabel('Infectious (% of population)')
axes[0].set_title('SIR Epidemic Curves')
axes[0].legend()

# Final epidemic size vs R0
R0_range = np.linspace(1.01, 5.0, 300)
R_inf = []
for R0 in R0_range:
    f = lambda x: x - 1 + np.exp(-R0 * x)
    R_inf.append(brentq(f, 1e-9, 1 - 1e-9))

axes[1].plot(R0_range, np.array(R_inf) * 100, linewidth=2, color='crimson')
axes[1].axvline(1, color='gray', linestyle='--', label='$\\mathcal{R}_0 = 1$ threshold')
for R0 in R0_values:
    f = lambda x: x - 1 + np.exp(-R0 * x)
    axes[1].plot(R0, brentq(f, 1e-9, 0.9999)*100, 'ko', markersize=5)
axes[1].set_xlabel('$\\mathcal{R}_0$')
axes[1].set_ylabel('Final epidemic size (% infected)')
axes[1].set_title('Final Epidemic Size vs. $\\mathcal{R}_0$')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.2 Effect of Vaccination

Vaccination reduces the effective susceptible fraction at $t=0$. If a fraction $v$ is vaccinated before the epidemic, the **effective reproduction number** is $\mathcal{R}_{\text{eff}} = \mathcal{R}_0 (1 - v)$. The epidemic is suppressed when $v \geq 1 - 1/\mathcal{R}_0$.

In [ ]:
R0 = 2.5
beta = R0 * gamma
vax_levels = [0.0, 0.2, 0.4, 1 - 1/R0, 0.7]

fig, ax = plt.subplots(figsize=(9, 4))
for v in vax_levels:
    S0 = (1 - v) - I0
    sol = solve_ivp(sir, t_span, [S0, I0, v], args=(beta, gamma),
                    t_eval=t_eval, rtol=1e-9)
    label = f'$v = {v:.0%}$' if v != 1 - 1/R0 else f'$v = {v:.1%}$ (herd immunity threshold)'
    ax.plot(sol.t, sol.y[1] * 100, label=label, linewidth=2)

ax.set_xlabel('Days')
ax.set_ylabel('Infectious (% of population)')
ax.set_title(f'SIR with Vaccination ($\\mathcal{{R}}_0 = {R0}$)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()